# 🪂 Lakeflow Designer (no-code ETL)

**Lakeflow Designer** is Databricks' no-code, node-based pipeline builder. Same engine as the SQL/Python pipelines you might write by hand — but you compose it by clicking + AI-assisted nodes, with a **live data preview at every step**.

We'll use it to build a small pipeline on top of the F1 silver tables you created in notebook 03.

> ⚠️ Lakeflow Designer is in **public preview** as of this workshop. If you cannot see it in your workspace, you may need a workspace admin to enable the preview in your workspace Previews area. 

## 🎯 What you'll build

A 3-node pipeline:

```text
  ┌────────────────────────────────────┐
  │ Source                             │
  │ main.default.f1_silver_race_results│
  └─────────────┬──────────────────────┘
                │  filter: finish_position <= 10
                ▼
  ┌────────────────────────────────────┐
  │ Transform                          │
  │ group by team                      │
  │ sum(points_earned) as season_points│
  │ count(*) as points_finishes        │
  └─────────────┬──────────────────────┘
                │
                ▼
  ┌────────────────────────────────────┐
  │ Sink                               │
  │ main.default.f1_team_points_finishes│
  └────────────────────────────────────┘
```

You'll see each row of data update live as you click through the nodes.

## 🛠 Steps

### 1️⃣ Open Lakeflow Designer

1. In the left sidebar, go to **+New → Visual data prep**.
2. Name it `F1 Team Points Finishes` in the top tab. 

![Create pipeline using Lakeflow Designer](./Images/Lakeflow_Designer_Create.png "Lakeflow Designer - Create")

### 2️⃣ Add the source node

1. Click **➕ Select a source → Browse existing**.
2. Pick `main.default.f1_silver_race_results`.
3. Once the data loads, it'll show you a **live preview** of the data. Confirm you see `race`, `finish_position`, `driver`, `team`, `points_earned`, etc.

### 3️⃣ Add a filter

1. From the source node, click **➕ → Filter**.
2. Set the expression: `finish_position <= 10`
3. Preview will show the filtered rows.

### 4️⃣ Add an aggregation

1. From the filter node, click **➕ → Aggregate**.
2. **Group by:** `team`
3. **Aggregations:**
   * `sum(points_earned)` → alias `season_points`
   * `count(points_earned)` → alias `points_finishes`
4. Click the node — preview should show one row per team.

### 5️⃣ Add the sink

1. From the aggregate node, click **➕ → Output**.
2. Catalog: `main`, Schema: `default`, Table: `f1_team_points_finishes`.
3. Click **▶️ Run**

### 6️⃣ How to run 

1. Click **▶️ Schedule** in the top right.
2. You can also add this to a job as a task. 

![Completed 3-node Lakeflow Designer pipeline](./Images/Lakeflow_Designer_DAG.png "Lakeflow Designer DAG")

In [0]:
%sql
-- Verify your Designer pipeline wrote the expected table.
-- Run this AFTER the pipeline finishes its first run.

SELECT
  team,
  season_points,
  points_finishes,
  ROUND(season_points * 1.0 / points_finishes, 2) AS avg_points_per_finish
FROM main.default.f1_team_points_finishes
ORDER BY season_points DESC;

## 🎯 Key takeaways

* You just built and scheduled a real ETL pipeline without writing any pipeline code.
* The output is a **standard Delta table** — every other tool in Databricks (SQL editor, dashboards, Genie, Power BI/Tableau via Lakeflow Connect) sees it like any other table.
* Unity Catalog permissions are **inherited automatically** — whoever can read the silver tables can read this one, no new grants needed.
* If you want this pipeline as code later, the engine underneath is the same as a SQL/Python Lakeflow pipeline — you can graduate to code when you outgrow the visual editor.

**Next:** `07_Dashboard_Builder.ipynb` — use Genie Code to spin up the workshop dashboard.